# Part 5 — Data Lake
## Task 5.1 — Cross-Format Queries using DuckDB

In [1]:
# Step 1 — Install and import duckdb
!pip install duckdb
import duckdb
print('DuckDB ready!')

DuckDB ready!


In [4]:
# Step 2 — Upload your 3 files
# Run this cell and upload customers.csv, orders.json, products.parquet
from google.colab import files
uploaded = files.upload()

Saving products.parquet to products (1).parquet
Saving orders.json to orders (1).json
Saving customers.csv to customers (2).csv


In [5]:
# Step 3 — Connect to DuckDB
con = duckdb.connect()
print('Connected to DuckDB!')

Connected to DuckDB!


In [6]:
# Q1: List all customers along with total number of orders they have placed
q1 = """
SELECT
    c.customer_id,
    c.name            AS customer_name,
    c.city,
    COUNT(o.order_id) AS total_orders
FROM read_csv_auto('customers.csv') AS c
LEFT JOIN read_json_auto('orders.json') AS o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name, c.city
ORDER BY total_orders DESC
"""
result_q1 = con.execute(q1).df()
print('Q1 — Customers with total orders:')
print(result_q1.to_string())

Q1 — Customers with total orders:
   customer_id  customer_name       city  total_orders
0      CUST048   Suresh Menon    Kolkata             6
1      CUST004     Neha Joshi     Mumbai             6
2      CUST011    Rohan Mehta       Pune             5
3      CUST006    Neha Sharma    Chennai             4
4      CUST012    Sneha Patel       Pune             4
5      CUST016   Vikram Singh     Mumbai             4
6      CUST032   Kiran Chopra    Chennai             4
7      CUST025    Aarav Desai    Kolkata             4
8      CUST002    Aditya Shah     Mumbai             3
9      CUST005    Divya Joshi     Mumbai             3
10     CUST020    Arjun Verma     Mumbai             3
11     CUST028   Pooja Sharma    Kolkata             3
12     CUST008      Amit Iyer     Mumbai             3
13     CUST017   Nikhil Mehta  Hyderabad             3
14     CUST026     Pooja Nair  Hyderabad             3
15     CUST037   Suresh Patel    Chennai             3
16     CUST038    Amit Tiwari  

In [7]:
# Q2: Find the top 3 customers by total order value
q2 = """
SELECT
    c.customer_id,
    c.name                AS customer_name,
    c.city,
    SUM(o.total_amount)   AS total_order_value,
    COUNT(o.order_id)     AS total_orders
FROM read_csv_auto('customers.csv') AS c
JOIN read_json_auto('orders.json') AS o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name, c.city
ORDER BY total_order_value DESC
LIMIT 3
"""
result_q2 = con.execute(q2).df()
print('Q2 — Top 3 customers by order value:')
print(result_q2.to_string())

Q2 — Top 3 customers by order value:
  customer_id customer_name     city  total_order_value  total_orders
0     CUST025   Aarav Desai  Kolkata            50331.0             4
1     CUST004    Neha Joshi   Mumbai            45527.0             6
2     CUST048  Suresh Menon  Kolkata            40629.0             6


In [8]:
# Q3: List all products purchased by customers from Bangalore
q3 = """
SELECT DISTINCT
    c.customer_id,
    c.name           AS customer_name,
    c.city,
    o.order_id,
    o.order_date,
    p.product_name,
    p.category
FROM read_csv_auto('customers.csv')   AS c
JOIN read_json_auto('orders.json')    AS o ON c.customer_id = o.customer_id
JOIN read_parquet('products.parquet') AS p ON o.order_id    = p.order_id
WHERE c.city = 'Bangalore'
ORDER BY c.customer_id
"""
result_q3 = con.execute(q3).df()
print('Q3 — Products bought by Bangalore customers:')
print(result_q3.to_string())

Q3 — Products bought by Bangalore customers:
  customer_id customer_name       city order_id order_date      product_name     category
0     CUST018     Neha Shah  Bangalore  ORD2082 2023-11-24       Denim Jeans     Clothing
1     CUST018     Neha Shah  Bangalore  ORD2034 2023-10-01         Desk Lamp   Home Decor
2     CUST018     Neha Shah  Bangalore  ORD2082 2023-11-24       Python Book        Books
3     CUST031  Rohan Pillai  Bangalore  ORD2031 2023-10-06         Desk Lamp   Home Decor
4     CUST031  Rohan Pillai  Bangalore  ORD2031 2023-10-06      Cotton Kurti     Clothing
5     CUST045  Aarav Sharma  Bangalore  ORD2068 2023-12-17  Wireless Earbuds  Electronics
6     CUST050   Sneha Mehta  Bangalore  ORD2033 2023-10-27       Denim Jeans     Clothing


In [9]:
# Q4: Join all 3 files — customer name, order date, product name, quantity
q4 = """
SELECT
    c.name         AS customer_name,
    c.city         AS customer_city,
    o.order_id,
    o.order_date,
    o.status       AS order_status,
    p.product_name,
    p.category,
    p.quantity,
    p.unit_price,
    o.total_amount
FROM read_csv_auto('customers.csv')   AS c
JOIN read_json_auto('orders.json')    AS o ON c.customer_id = o.customer_id
JOIN read_parquet('products.parquet') AS p ON o.order_id    = p.order_id
ORDER BY o.order_date
"""
result_q4 = con.execute(q4).df()
print('Q4 — Full join: customer name, order date, product name, quantity:')
print(result_q4.to_string())

Q4 — Full join: customer name, order date, product name, quantity:
    customer_name customer_city order_id order_date order_status         product_name       category  quantity  unit_price  total_amount
0     Rohan Gupta       Chennai  ORD2046 2023-01-01    cancelled            Face Wash  Personal Care         5         299          1398
1     Arjun Singh          Pune  ORD2004 2023-01-07      shipped     Basmati Rice 5kg        Grocery         4         450          5596
2      Pooja Nair     Hyderabad  ORD2098 2023-01-10    delivered     Wireless Earbuds    Electronics         1        2499          4649
3     Amit Tiwari     Hyderabad  ORD2036 2023-01-15      shipped             Yoga Mat         Sports         3         699         11146
4     Rohan Mehta          Pune  ORD2096 2023-02-03    cancelled     Wireless Earbuds    Electronics         1        2499          1598
5      Pooja Nair     Hyderabad  ORD2007 2023-02-05      shipped  Mechanical Keyboard    Electronics         2 